# 07 - World Models & Environment Modeling

## Scenario: Mental Maps of the Northstar VPC

If you give an agent a `restart_service` tool and tell it "Fix the API", it might restart the API gateway, which brings down the entire site. 

Advanced agents need a **World Model**—a structured understanding of the environment they operate in, including dependencies, rules, and blast radii. Instead of just giving tools, we give the agent a Knowledge Graph.

In [1]:
# 1. Initialization and Mock Fallback
import os
import sys

if os.environ.get("OPENAI_API_KEY"):
    from openai import OpenAI
    client = OpenAI()
    print("✅ Using real OpenAI API.")
else:
    print("⚠️ No OPENAI_API_KEY found. Falling back to MockOpenAI...")
    sys.path.append(os.path.abspath("../../.."))
    try:
        from awsome_agents.mock_openai import MockOpenAI
        client = MockOpenAI()
    except ImportError:
        print("Failed to import MockOpenAI. Ensure you are running from the repository root.")

⚠️ No OPENAI_API_KEY found. Falling back to MockOpenAI...
🔧 Initialized MockOpenAI Client (Network requests disabled)


## 1. Defining the Environment Graph

We define the architecture dependencies in Python. The Agent can query this graph *before* making destructive changes.

In [2]:
# A simple adjacency list representing our VPC microservices
infrastructure_graph = {
    "api-gateway": ["checkout-service", "auth-service"],
    "checkout-service": ["payment-db", "redis-cache"],
    "auth-service": ["user-db"]
}

def get_downstream_dependencies(service: str) -> list:
    """Tool: Allows the agent to see what will break if a service goes down."""
    deps = infrastructure_graph.get(service, [])
    print(f"  🔍 [Tool] Querying Graph: {service} dependencies -> {deps}")
    return deps


## 2. Using the World Model

In [3]:
def evaluate_restart(target_service: str):
    print(f"🧠 [Agent] I need to restart {target_service}.")
    print(f"🧠 [Agent] Let me check my World Model to see the blast radius.")
    
    deps = get_downstream_dependencies(target_service)
    
    if len(deps) > 0:
        print(f"🚨 [Agent Safety Protocol] WARNING: Restarting {target_service} will take down {', '.join(deps)}!")
        print(f"🚨 [Agent Safety Protocol] Action blocked. Escalating to human.")
    else:
        print(f"✅ [Agent] No downstream dependencies. Safe to restart {target_service}.")

# 1. Attempting to restart a leaf node (safe)
evaluate_restart("redis-cache")

print("\n------------------------\n")

# 2. Attempting to restart the root node (dangerous)
evaluate_restart("api-gateway")


🧠 [Agent] I need to restart redis-cache.
🧠 [Agent] Let me check my World Model to see the blast radius.
  🔍 [Tool] Querying Graph: redis-cache dependencies -> []
✅ [Agent] No downstream dependencies. Safe to restart redis-cache.

------------------------

🧠 [Agent] I need to restart api-gateway.
🧠 [Agent] Let me check my World Model to see the blast radius.
  🔍 [Tool] Querying Graph: api-gateway dependencies -> ['checkout-service', 'auth-service']
🚨 [Agent Safety Protocol] WARNING: Restarting api-gateway will take down checkout-service, auth-service!
🚨 [Agent Safety Protocol] Action blocked. Escalating to human.


## Checkpoint

**1. What is a 'World Model' in Agentic AI?**
- A) A 3D simulation of the earth.
- B) A structured representation (like a graph or rule engine) of the environment, allowing the agent to understand dependencies and consequences *before* acting.
- C) A global translation model.
- D) A database of all internet websites.
